# Center Crop Accuracy Test

Measures how much the model relies on image edges for classification.

For each radius fraction, pixels **outside** the masked region are replaced with either:
- **zero** — constant zero (black in un-normalized space)
- **noise** — Gaussian random noise

The mask is applied **after** the standard timm transform (in normalized tensor space).  
Top-1 accuracy vs ground-truth labels is computed for each radius on the full val set.

In [ ]:
import torch
import timm
import timm.data
import numpy as np
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME    = "vit_large_patch14_clip_224.openai_ft_in1k"
IMAGENET_ROOT = "/data/imagenet/extracted"   # path with val/ subdir
BATCH_SIZE    = 64
NUM_WORKERS   = 4
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

# Radius fractions to test: fraction of the image half-size kept
# 1.0 = full image (no masking), 0.2 = keep only the central 20%
RADIUS_FRACS  = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# Mask shapes to test
MASK_SHAPES   = ["circular", "square"]

# Fill modes for masked-out pixels
FILL_MODES    = ["zero", "noise"]
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# Load model
model = timm.create_model(MODEL_NAME, pretrained=True)
model.eval().to(DEVICE)
for p in model.parameters():
    p.requires_grad_(False)

print(f"Model : {MODEL_NAME}")
print(f"Device: {DEVICE}")

In [ ]:
# Build val dataloader
data_config   = timm.data.resolve_model_data_config(model)
val_transform = timm.data.create_transform(**data_config, is_training=False)

val_dataset = ImageFolder(f"{IMAGENET_ROOT}/val", transform=val_transform)
val_loader  = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Val images : {len(val_dataset):,}")
print(f"Val batches: {len(val_loader):,}")

In [ ]:
def build_mask(H: int, W: int, radius_frac: float, shape: str) -> torch.Tensor:
    """Return a boolean mask of shape [H, W].

    True  → keep pixel (inside the center region)
    False → mask out  (outside the center region)

    Args:
        H, W        : image height and width in pixels.
        radius_frac : fraction of half the image size to keep (1.0 = full image).
        shape       : "circular" or "square".
    """
    cy, cx = H / 2.0, W / 2.0

    if shape == "circular":
        radius  = radius_frac * min(H, W) / 2.0
        ys = torch.arange(H).float().unsqueeze(1)  # [H, 1]
        xs = torch.arange(W).float().unsqueeze(0)  # [1, W]
        dist = ((ys - cy) ** 2 + (xs - cx) ** 2).sqrt()
        return dist <= radius
    else:  # square
        rh = radius_frac * H / 2.0
        rw = radius_frac * W / 2.0
        ys = torch.arange(H).float()
        xs = torch.arange(W).float()
        in_y = (ys >= cy - rh) & (ys < cy + rh)  # [H]
        in_x = (xs >= cx - rw) & (xs < cx + rw)  # [W]
        return in_y.unsqueeze(1) & in_x.unsqueeze(0)  # [H, W]


def apply_center_mask(
    images: torch.Tensor,
    radius_frac: float,
    shape: str = "circular",
    fill: str = "zero",
) -> torch.Tensor:
    """Apply a center-keep mask to a batch of normalized images.

    Args:
        images      : [B, C, H, W] normalized float tensor.
        radius_frac : fraction of half image size to keep.
        shape       : "circular" or "square".
        fill        : "zero" or "noise" for masked-out pixels.

    Returns:
        Masked images [B, C, H, W].
    """
    if radius_frac >= 1.0:
        return images  # no masking

    B, C, H, W = images.shape
    mask = build_mask(H, W, radius_frac, shape).to(images.device)  # [H, W]
    mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]

    if fill == "zero":
        fill_values = torch.zeros_like(images)
    else:  # noise
        fill_values = torch.randn_like(images)

    return torch.where(mask.expand_as(images), images, fill_values)


# Quick sanity check on mask shapes
dummy = torch.zeros(1, 3, 224, 224)
for shape in MASK_SHAPES:
    masked = apply_center_mask(dummy, 0.5, shape=shape, fill="zero")
    kept   = (masked.abs().sum(1) == 0).float().mean().item()
    print(f"{shape:10s} r=0.5 → {kept*100:.1f}% pixels zeroed out")

In [ ]:
@torch.no_grad()
def evaluate_masked(
    model,
    val_loader: DataLoader,
    radius_frac: float,
    shape: str,
    fill: str,
    device: str,
) -> float:
    """Return top-1 accuracy for one (radius, shape, fill) combination."""
    correct = 0
    total   = 0
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        images = apply_center_mask(images, radius_frac, shape=shape, fill=fill)
        preds  = model(images).argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total   += images.size(0)
    return correct / total * 100

In [ ]:
# ── Run sweep ─────────────────────────────────────────────────────────────────
# results[shape][fill] = list of accuracies (one per radius in RADIUS_FRACS)
results = {s: {f: [] for f in FILL_MODES} for s in MASK_SHAPES}

for shape in MASK_SHAPES:
    for fill in FILL_MODES:
        print(f"\n[{shape} / {fill}]")
        for r in RADIUS_FRACS:
            acc = evaluate_masked(model, val_loader, r, shape, fill, DEVICE)
            results[shape][fill].append(acc)
            print(f"  radius={r:.1f}  top-1 acc = {acc:.2f}%")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(MASK_SHAPES), figsize=(7 * len(MASK_SHAPES), 5), sharey=True)
if len(MASK_SHAPES) == 1:
    axes = [axes]

colors = {"zero": "steelblue", "noise": "darkorange"}
markers = {"zero": "o", "noise": "s"}

for ax, shape in zip(axes, MASK_SHAPES):
    for fill in FILL_MODES:
        ax.plot(
            RADIUS_FRACS,
            results[shape][fill],
            color=colors[fill],
            marker=markers[fill],
            linewidth=2,
            label=f"fill={fill}",
        )
        # Annotate the full-image accuracy
        full_acc = results[shape][fill][-1]
        ax.axhline(full_acc, color=colors[fill], linestyle="--", alpha=0.3)

    ax.set_title(f"{shape.capitalize()} mask — {MODEL_NAME}", fontsize=11)
    ax.set_xlabel("Radius fraction (1.0 = full image)", fontsize=10)
    ax.set_ylabel("Top-1 accuracy (%)", fontsize=10)
    ax.set_xticks(RADIUS_FRACS)
    ax.set_xticklabels([f"{r:.1f}" for r in RADIUS_FRACS], fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.legend(fontsize=9)

plt.suptitle("Model accuracy vs center mask radius", fontsize=13)
plt.tight_layout()
plt.savefig("center_crop_accuracy.png", dpi=150)
plt.show()
print("Plot saved to center_crop_accuracy.png")

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────────
print(f"{'Radius':>8}  ", end="")
for shape in MASK_SHAPES:
    for fill in FILL_MODES:
        print(f"{shape[:4]}/{fill[:5]:>12}", end="  ")
print()
print("-" * (10 + 16 * len(MASK_SHAPES) * len(FILL_MODES)))

for i, r in enumerate(RADIUS_FRACS):
    print(f"{r:>8.1f}  ", end="")
    for shape in MASK_SHAPES:
        for fill in FILL_MODES:
            acc = results[shape][fill][i]
            print(f"{acc:>14.2f}%", end="  ")
    print()